In [1]:
import pandas as pd
import numpy as np
import sklearn

from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
import seaborn as sns



ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df1=pd.read_csv('spam_ham_india.csv')
df1.rename(columns={'Msg':'text', 'Label':'target'}, inplace=True)


df1['target']=encoder.fit_transform(df1['target'])


In [ ]:
df2=pd.read_csv('cleaned_sms_spam.csv')
df2.drop(columns=['Unnamed: 0'], inplace=True)
print(df1['target'].value_counts())
print(df2['target'].value_counts())
df=pd.concat([df1,df2], ignore_index=True)
print(df.shape)
df.sample(20)

In [ ]:
df['text'].apply(lambda x: isinstance(x,(int, float))).value_counts()

In [ ]:
# df.drop(index=df[df['text'].apply(lambda x: isinstance(x,(int, float)))].index, inplace=True)
# df['text'].apply(lambda x: isinstance(x,(int, float))).value_counts()


In [ ]:
df.drop(index=df[df['text'].apply(lambda x: isinstance(x,(int, float)))].index, inplace=True)

In [ ]:
df.head()

In [ ]:
df['target'].value_counts()

In [ ]:
plt.pie(df['target'].value_counts(), labels=['ham','spam'], autopct='%0.2f')
plt.show()

In [ ]:
df.to_csv("data.csv", index=False, encoding='utf-8')

In [ ]:

df['num_characters'] = df['text'].apply(lambda x: len((str(x).replace(" ", ""))))
df.head()

In [ ]:
df['num_words']=df['text'].apply(lambda x: len(nltk.word_tokenize(str(x))))
df.head()

In [ ]:
df['num_sentences']=df['text'].apply(lambda x: len(nltk.sent_tokenize(str(x))))
df.head()


In [ ]:
df[df['target']==0][['num_characters','num_words','num_sentences']].describe()


In [ ]:
df[df['target']==1][['num_characters','num_words','num_sentences']].describe()


In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(df[df['target']==0]['num_characters'])
sns.histplot(df[df['target']==1]['num_characters'])

In [ ]:
sns.histplot(df[df['target']==0]['num_words'])
sns.histplot(df[df['target']==1]['num_words'])

In [ ]:
sns.pairplot(df,hue='target')

In [ ]:
sns.heatmap(df.drop(columns=['text']).corr(), annot=True)

In [ ]:
#here we can observe num of words, num of chars and num of sentences are highly correlated with each other. so we can drop any 2 of them and keep one feature for our model. we will keep num_characters and drop the other two features as it has highest correlation with target variable.

## Text Preprocessing
1. lower case
2. tokenization
3. removing special characters
4. removing stopwords and punctuations
5. removing words with same meaning (stemming)

In [ ]:
from nltk.corpus import stopwords
stopwords.words('english')


In [ ]:
import string
string.punctuation

In [ ]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()
ps.stem('playing')

In [ ]:
def text_pp(text):
    text=text.lower()
    text=nltk.word_tokenize(text)
    y=[]
    for i in text:
        if i.isalnum():
            y.append(i)
    text=y[:]
    y.clear()
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    
    text=y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
text_pp('hi my name is arham, and i love to play cricket. what do you like?')

In [ ]:
text_pp('1047956&%')

In [ ]:
df['transformed_text']=df['text'].apply(text_pp)
df.head()

In [ ]:
from wordcloud import WordCloud
wc=WordCloud(width=700, height=700, min_font_size=10, background_color='white')
spam_wc=wc.generate(df[df['target']==1]['transformed_text'].str.cat(sep=" "))
plt.imshow(spam_wc)

In [ ]:
ham_wc=wc.generate(df[df['target']==0]['transformed_text'].str.cat(sep=" "))
plt.imshow(ham_wc)

In [ ]:
df.head()

In [ ]:
df[df['target']==1]['transformed_text']


In [ ]:
df[df['target']==1]['transformed_text'].tolist()

In [ ]:
spam_corpus=[]
for msg in df[df['target']==1]['transformed_text'].tolist():
    for word in msg.split():
        spam_corpus.append(word)
print(spam_corpus)

#this gives us a corpus of words that appear in spam messages

In [ ]:
from collections import Counter
Counter(spam_corpus).most_common(30)

In [ ]:
pd.DataFrame(Counter(spam_corpus).most_common(30))


In [ ]:
sns.barplot(x=pd.DataFrame(Counter(spam_corpus).most_common(30))[0],y=pd.DataFrame(Counter(spam_corpus).most_common(30))[1])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
ham_corpus=[]
for msg in df[df['target']==0]['transformed_text'].tolist():
    for word in msg.split():
        ham_corpus.append(word)
print(ham_corpus)

In [ ]:
sns.barplot(x=pd.DataFrame(Counter(ham_corpus).most_common(30))[0],y=pd.DataFrame(Counter(ham_corpus).most_common(30))[1])
plt.xticks(rotation='vertical')
plt.show()

## Text Vectorisation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()


In [ ]:
X=tfidf.fit_transform(df['transformed_text']).toarray()
print(X.shape)
y=df['target']

In [ ]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
gnb=GaussianNB()
mnb=MultinomialNB()
bnb=BernoulliNB()

from sklearn.metrics import accuracy_score, confusion_matrix, precision_score

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=2)


In [ ]:
gnb.fit(X_train,y_train)
y_pred1=gnb.predict(X_test)
print(accuracy_score(y_pred1,y_test))
print(confusion_matrix(y_pred1,y_test))
print(precision_score(y_pred1,y_test))



In [ ]:
mnb.fit(X_train,y_train)
y_pred2=mnb.predict(X_test)
print(accuracy_score(y_pred2,y_test))
print(confusion_matrix(y_pred2,y_test))
print(precision_score(y_pred2,y_test))

In [ ]:
bnb.fit(X_train,y_train)
y_pred3=bnb.predict(X_test)
print(accuracy_score(y_pred3,y_test))
print(confusion_matrix(y_pred3,y_test))
print(precision_score(y_pred3,y_test))

In [ ]:

data1 = {
    "gnb_tfif": [accuracy_score(y_pred1, y_test), precision_score(y_pred1,y_test)],
    "mnb_tfif": [accuracy_score(y_pred2, y_test), precision_score(y_pred2,y_test)],
    "bnb_tfif": [accuracy_score(y_pred3, y_test), precision_score(y_pred3,y_test)]
}
pd.DataFrame(data1,index=['Accuracy','Precision']).T.plot(kind='bar',color=['magenta','yellow'])
plt.show()

# Above graph shows the model statistics for the Gaussian, Multinomial and Bernoulli's distribution for Naive-Baye's classifier when TF-IDF Vectorisation is used.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()

X1=cv.fit_transform(df['transformed_text']).toarray()
y1=df['target']

In [ ]:
X1_train,X1_test,y1_train,y1_test=train_test_split(X1,y1,test_size=0.2,random_state=2)


In [ ]:
gnb.fit(X1_train,y1_train)
y1_pred1=gnb.predict(X_test)
print(accuracy_score(y1_pred1,y1_test))
print(confusion_matrix(y1_pred1,y1_test))
print(precision_score(y1_pred1,y1_test))

In [ ]:
mnb.fit(X1_train,y1_train)
y1_pred2=mnb.predict(X1_test)
print(accuracy_score(y1_pred2,y1_test))
print(confusion_matrix(y1_pred2,y1_test))
print(precision_score(y1_pred2,y1_test))

In [ ]:
bnb.fit(X1_train,y1_train)
y1_pred3=bnb.predict(X1_test)
print(accuracy_score(y1_pred3,y1_test))
print(confusion_matrix(y1_pred3,y1_test))
print(precision_score(y1_pred3,y1_test))

In [ ]:
data2 = {
    "gnb_cv": [accuracy_score(y1_pred1, y1_test), precision_score(y1_pred1,y1_test)],
    "mnb_cv": [accuracy_score(y1_pred2, y1_test), precision_score(y1_pred2,y1_test)],
    "bnb_cv": [accuracy_score(y1_pred3, y1_test), precision_score(y1_pred3,y1_test)]
}
pd.DataFrame(data2,index=['Accuracy','Precision']).T.plot(kind='bar',color=['cyan','yellow'])
plt.show()

From the above two tables, its clear that Multinomial Naive-Baye's trained on Bag of Words Vectorization is giving best results of all six.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier


In [ ]:
nb=MultinomialNB()
rfc=RandomForestClassifier(n_estimators=50,random_state=2)
etc=ExtraTreesClassifier(n_estimators=50,random_state=2)
abc=AdaBoostClassifier(n_estimators=50,random_state=2)


In [ ]:
clfs={
    'Naive-Bayes':nb,
    'RandomForestClassifier':rfc,
    'ExtraTreeClassifier':etc,
    'AdaBoost':abc
}
def predic(classifier,X_train,X_test,y_train,y_test):
    classifier.fit(X_train,y_train)
    y_pred=classifier.predict(X_test)
    acc=accuracy_score(y_test,y_pred)
    prec=precision_score(y_test,y_pred)
    return acc,prec


accuracy_sc_tfidf=[]
accuracy_sc_bow=[]
precision_sc_tfidf=[]
precision_sc_bow=[]

for clf in clfs.values():
    accu,preci=predic(clf,X_train,X_test,y_train,y_test)
    accuracy_sc_tfidf.append(accu)
    precision_sc_tfidf.append(preci)
    accu,preci=predic(clf,X1_train,X1_test,y1_train,y1_test)
    accuracy_sc_bow.append(accu)
    precision_sc_bow.append(preci)



performance_df=pd.DataFrame({'Model':clfs.keys(),'Accuracy TFIDF':accuracy_sc_tfidf,'Precision TFIDF':precision_sc_tfidf,'Accuracy BOW':accuracy_sc_bow,'Precision BOW':precision_sc_bow})


In [ ]:
performance_df.head()

In [ ]:
performance_df.index=clfs.keys()
a=performance_df.head().plot(kind='bar',color=['skyblue','yellow','red','white'])
plt.show()

# In a side by side comparision, we can observe that the TFIDF Vectorisation is giving better overall results as compared to Bag of Words.

In [ ]:
tfidf_=TfidfVectorizer(max_features=2000)

In [ ]:
X2 = tfidf_.fit_transform(df['transformed_text']).toarray()
print(X2.shape)
y2 = df['target']

In [ ]:
X2_train,X2_test,y2_train,y2_test=train_test_split(X2,y2,test_size=0.2,random_state=42)

In [ ]:
X2_train,X2_test,y2_train,y2_test = train_test_split(X2,y2,test_size=0.2,random_state=40)
accuracy_mp=[]
precision_mp=[]
for clf in clfs.values():
    accu,preci=predic(clf,X2_train,X2_test,y2_train,y2_test)
    accuracy_mp.append(accu)
    precision_mp.append(preci)
performance_df2=pd.DataFrame({'Model':clfs.keys(),'AccuracyTFIDF':accuracy_sc_tfidf,'PrecisionTFIDF':precision_sc_tfidf,'AccuracyTFIDF3000':accuracy_mp,'PrecisionTFIDF3000':precision_mp})
performance_df2.head()

In [ ]:
from sklearn.ensemble import VotingClassifier
voting=VotingClassifier(estimators=[('nb',nb),('rf',rfc),('et',etc),('ab',abc)],voting='hard')
voting.fit(X2_train,y2_train)
y2_pred=voting.predict(X2_test)
print('Accuracy= ',accuracy_score(y2_pred,y2_test))
print('Precision= ',precision_score(y2_pred,y2_test))


In [ ]:
estimators=[('rf',rfc),('et',etc),('ab',abc)]
final_estimator=MultinomialNB()

In [ ]:
from sklearn.ensemble import StackingClassifier

In [ ]:
clf=StackingClassifier(estimators=estimators,final_estimator=final_estimator)
clf.fit(X2_train,y2_train)


In [ ]:
y3_pred=clf.predict(X2_test)
print('Accuracy= ',accuracy_score(y2_test,y3_pred))
print('Precision= ',precision_score(y2_test,y3_pred))



# Out of all of these, Random Forest seems to be giving the best output overall so we will be using it

In [ ]:
# import pickle
# pickle.dump(tfidf_,open('Vectorizer.pkl',"wb"))
# pickle.dump(rfc,open('model.pkl',"wb"))

In [ ]:
#Versions
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("sklearn", sklearn.__version__)
print("seaborn", sns.__version__)